# 日本語の文分割をライブラリで行って比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/04_japanese_tokenization/sentence_split_libs.ipynb)

NLP クックブック 04-24 ／ 日本語の形態素解析と文分割

日本語の文章を文に区切りたい。括弧の中の句点、改行、絵文字や顔文字で終わる文など、単純な規則では失敗する場合も扱いたい。

In [1]:
!pip install -q ja-sentence-segmenter pysbd ginza ja_ginza

In [2]:
import functools
import re
import pysbd
import spacy
from ja_sentence_segmenter.common.pipeline import make_pipeline
from ja_sentence_segmenter.concatenate.simple_concatenator import concatenate_matching
from ja_sentence_segmenter.normalize.neologd_normalizer import normalize
from ja_sentence_segmenter.split.simple_splitter import split_newline, split_punctuation

# 1. 正規表現だけ
def split_regex(text):
    return [s for s in re.split(r"(?<=[。！？!?])|\n", text) if s and s.strip()]

# 2. ja-sentence-segmenter: 正規化 → 改行で分割 → 句読点で分割 (括弧内は分割しない) → 分割しすぎた箇所を連結、を組み合わせる
split_punc = functools.partial(split_punctuation, punctuations=r"。!?")
concat_tail = functools.partial(concatenate_matching, former_matching_rule=r"^(?P<result>.+)(て)$", remove_former_matched=False)
jss = make_pipeline(normalize, split_newline, concat_tail, split_punc)

# 3. pySBD: 多言語対応の規則ベース  4. GiNZA: 係り受け解析の結果から文境界を決める
sbd = pysbd.Segmenter(language="ja", clean=False)
nlp = spacy.load("ja_ginza")

splitters = {
    "正規表現": split_regex,
    "ja-sentence-segmenter": lambda t: list(jss(t)),
    "pySBD": lambda t: [s.strip() for s in sbd.segment(t)],
    "GiNZA": lambda t: [s.text.strip() for s in nlp(t.replace("\n", "")).sents],
}

text = ("彼は「もう帰る。明日も早いから」と言った。本当ですか！？\n"
        "はい（たぶん。）そうです。今日は楽しかった😊また行こうね(^^)\n"
        "値段は1.5倍になった。詳しくはhttps://example.com/a.htmlを見て\n"
        "ください。最後は句点なし")
for name, f in splitters.items():
    sents = f(text)
    print(f"[{name}] {len(sents)} 文")
    for s in sents:
        print("   ", s)

[正規表現] 11 文
    彼は「もう帰る。
    明日も早いから」と言った。
    本当ですか！
    ？
    はい（たぶん。
    ）そうです。
    今日は楽しかった😊また行こうね(^^)
    値段は1.5倍になった。
    詳しくはhttps://example.com/a.htmlを見て
    ください。
    最後は句点なし
[ja-sentence-segmenter] 8 文
    彼は「もう帰る。明日も早いから」と言った。
    本当ですか!
    ?
    はい(たぶん。)そうです。
    今日は楽しかった😊また行こうね(^^)
    値段は1.5倍になった。
    詳しくはhttps://example.com/a.htmlを見てください。
    最後は句点なし
[pySBD] 9 文
    彼は「もう帰る。明日も早いから」と言った。
    本当ですか！
    ？
    はい（たぶん。）そうです。
    今日は楽しかった😊また行こうね(^^)
    値段は1.5倍になった。
    詳しくはhttps://example.com/a.htmlを見て
    ください。
    最後は句点なし
[GiNZA] 9 文
    彼は「もう帰る。明日も早いから」と言った。
    本当ですか！？
    はい（たぶん。）
    そうです。
    今日は楽しかった😊
    また行こうね(^^)
    値段は1.5倍になった。
    詳しくはhttps://example.com/a.htmlを見てください。
    最後は句点なし


In [3]:
# 大量の文章を処理するときの速度
import time

long_text = "今日は良い天気です。散歩に行きましょう！どこへ行きますか？「公園がいい。近いから」と彼は言った。" * 200
for name, f in splitters.items():
    t = time.perf_counter()
    n = len(f(long_text))
    print(f"{name:22} {n:5} 文  {time.perf_counter() - t:6.2f} 秒")

正規表現                    1000 文    0.00 秒
ja-sentence-segmenter    800 文    0.00 秒
pySBD                    800 文    0.05 秒
GiNZA                    800 文    2.49 秒


- 単純な正規表現は、括弧内の句点で切ってしまい、絵文字や顔文字で終わる文は切れない。小数点や URL の中のピリオドは、全角の句点だけを区切りにすれば避けられる。
- ja-sentence-segmenter は、分割と連結の規則を自分で組み替えられる。行の途中で折り返された文章 (メールや PDF 由来のテキスト) をつなぎ直す処理も書ける。
- GiNZA は句点のない文末 (絵文字、顔文字の後ろ) も区切れるが、解析を伴うので 2 桁以上遅い。括弧内で切ってしまうこともある。
- SNS や口コミのような崩れた文章向けには、学習ベースの bunkai がある (執筆時点では依存ライブラリ emoji の新しい版と互換性がなく、そのままでは動かなかった)。
- 文分割の誤りは後段のすべての処理に響く。自分のデータの数十文で、どこで切れているかを必ず目で確かめる。